# NB30 · Networking, HTTP & APIs

`advanced/` · 20 questions · the last one is a story project

Almost every real program talks to another program over a network: a
weather service, a payment provider, a browser. This notebook shows what
happens underneath: raw **sockets** moving bytes, **HTTP** (methods, status
codes, headers, JSON bodies) on top of them, and the standard-library tools
that speak it: `urllib.parse` builds and splits URLs, `urllib.request`
sends requests, `http.server` answers them.

**Everything runs on your own computer.** Servers listen on `127.0.0.1`
(this machine only) on port `0`, which asks the operating system for any
free port, and they run in a background thread (NB27). Every request has a
**timeout**, because without one a request can wait forever. Every server
is shut down at the end of its cell. Port numbers change on each run, so
nothing prints them, and nothing ever touches the internet.

In real projects you will usually install `requests` or `httpx`. They wrap
exactly what you practise here: they build query strings for you
(`params=`), encode and decode JSON (`json=`, `.json()`), set sensible
defaults, reuse connections, and can retry. Knowing the layer below is what
lets you debug them when a request misbehaves. The exercises stay in the
standard library.

Where a question needs a server, the starting code already contains it
(a small `Handler` class); read it, then write the client side.

## You'll need

- NB27: threads and asyncio
- NB13: JSON

## You'll practise

- How HTTP works: methods, status codes, headers, JSON bodies, REST conventions
- Sockets: a tiny echo server and client on localhost in a thread
- http.server: serving a small JSON API inside the notebook
- urllib.request and urllib.parse: query strings, POST bodies, errors
- Timeouts, retries with backoff, handling 4xx vs 5xx
- What requests and httpx add (explained; exercises stay standard-library)

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Questions get longer as you go: one-liners first, a small story project last. Every solution only uses what this notebook and the ones before it teach.

---

## Warm-up · one line each

### Q01 · Real-life

Build the full search URL from `base` and `params` with `urllib.parse`
(spaces and special characters must be escaped for you) and print it.

**Expected output:**

```text
https://shop.example.com/search?q=red+shoes&size=42&sort=price
```

<details><summary><b>Solution</b></summary>

```python
from urllib.parse import urlencode
print(base + "?" + urlencode(params))
```

</details>

In [ ]:
base = "https://shop.example.com/search"
params = {"q": "red shoes", "size": 42, "sort": "price"}

# your code here


### Q02 · Real-life

Take `url` apart with `urllib.parse`. Print the host name, the port as a
number, the list of `tag` values, and the `page` as a number, one per line.

**Expected output:**

```text
api.example.com
8443
['sale', 'new']
3
```

<details><summary><b>Solution</b></summary>

```python
from urllib.parse import parse_qs, urlsplit
parts = urlsplit(url)
query = parse_qs(parts.query)
print(parts.hostname)
print(parts.port)
print(query["tag"])
print(int(query["page"][0]))
```

</details>

In [ ]:
url = "https://api.example.com:8443/v2/search?tag=sale&tag=new&page=3"

# your code here


### Q03 · Output-match

A monitoring script logs these HTTP status codes. For each one print the
code, its official name from `http.HTTPStatus`, and `True` if it is a
server error (5xx, where retrying might help) or `False` otherwise, all on
one line.

**Expected output:**

```text
200 OK False
404 Not Found False
429 Too Many Requests False
503 Service Unavailable True
```

<details><summary><b>Solution</b></summary>

```python
from http import HTTPStatus
for code in codes:
    print(code, HTTPStatus(code).phrase, code // 100 == 5)
```

</details>

In [ ]:
codes = [200, 404, 429, 503]

# your code here


### Q04 · Real-life

The file name `fish & chips.txt` goes into two URLs: as the last part of a
**path**, and as the value of the `q` parameter in a **query string**. Print
the escaped path piece, then the whole escaped query string.

**Expected output:**

```text
fish%20%26%20chips.txt
q=fish+%26+chips.txt
```

<details><summary><b>Solution</b></summary>

```python
from urllib.parse import quote, urlencode
print(quote(name))
print(urlencode({"q": name}))
```

</details>

In [ ]:
name = "fish & chips.txt"

# your code here


### Q05 · Build

Build (but do not send) a `urllib.request.Request` that POSTs `order` as
JSON to `url`: the body is the JSON text as bytes, with a `Content-Type`
of `application/json`. Print the method, the content type read back from
the request, and the body decoded to text.

**Expected output:**

```text
POST
application/json
{"item": "apple", "qty": 3}
```

<details><summary><b>Solution</b></summary>

```python
import urllib.request
request = urllib.request.Request(url, data=json.dumps(order).encode(),
                                 headers={"Content-Type": "application/json"})
print(request.get_method())
print(request.get_header("Content-type"))
print(request.data.decode())
```

</details>

In [ ]:
import json
url = "http://127.0.0.1/api/orders"
order = {"item": "apple", "qty": 3}

# your code here


---

## Combine · 2 to 4 lines

### Q06 · Puzzle

An HTTP body travels as bytes, and the `Content-Length` header counts
**bytes**, not characters. Turn `reply` into JSON text that keeps accented
letters as they are (not escaped), then print the number of characters, the
number of bytes after encoding to UTF-8, and the header line to send.

**Expected output:**

```text
characters: 30
bytes: 32
Content-Length: 32
```

<details><summary><b>Solution</b></summary>

```python
text = json.dumps(reply, ensure_ascii=False)
body = text.encode("utf-8")
print("characters:", len(text))
print("bytes:", len(body))
print("Content-Length: " + str(len(body)))
```

</details>

In [ ]:
import json
reply = {"name": "Café Zoë", "qty": 2}

# your code here


### Q07 · Build

A server is already waiting on `address` (see the starting code). Connect
to it with a socket and a 2-second timeout, send the bytes `hello server`,
read the reply, and print it decoded. Close the connection and the
`listener` at the end.

**Expected output:**

```text
HELLO SERVER
```

<details><summary><b>Solution</b></summary>

```python
client = socket.create_connection(address, timeout=2)
client.sendall(b"hello server")
print(client.recv(1024).decode())
client.close()
listener.close()
```

</details>

In [ ]:
import socket
import threading

listener = socket.socket()
listener.bind(("127.0.0.1", 0))
listener.listen()
address = listener.getsockname()

def serve_once():
    conn, _ = listener.accept()
    with conn:
        conn.sendall(conn.recv(1024).upper())

threading.Thread(target=serve_once, daemon=True).start()

# your code here


### Q08 · Real-life

The server at `base` answers `/api/products` with a JSON list. Fetch it
with a 2-second timeout, then print how many products came back and their
total price. Shut the server down at the end.

**Expected output:**

```text
Products: 3
Total price: 24.75
```

<details><summary><b>Solution</b></summary>

```python
with urllib.request.urlopen(base + "/api/products", timeout=2) as response:
    products = json.load(response)
print("Products:", len(products))
print("Total price:", sum(item["price"] for item in products))
server.shutdown()
server.server_close()
```

</details>

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/api/products":
            self.send_json(200, [{"name": "pen", "price": 1.5},
                                 {"name": "notebook", "price": 4.25},
                                 {"name": "bag", "price": 19.0}])
        else:
            self.send_json(404, {"error": "not found"})

    def send_json(self, status, data, headers=None):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, "http://127.0.0.1:" + str(server.server_address[1])

server, base = start_server(Handler)

# your code here


### Q09 · Real-life

Call `/api/health` and print the status code, the `Content-Type` header,
the `X-Api-Version` header and the `status` field from the JSON body, one
per line. Shut the server down at the end.

**Expected output:**

```text
200
application/json
2.3
ok
```

<details><summary><b>Solution</b></summary>

```python
with urllib.request.urlopen(base + "/api/health", timeout=2) as response:
    print(response.status)
    print(response.headers["Content-Type"])
    print(response.headers["X-Api-Version"])
    print(json.load(response)["status"])
server.shutdown()
server.server_close()
```

</details>

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_json(200, {"status": "ok"}, {"X-Api-Version": "2.3"})

    def send_json(self, status, data, headers=None):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, "http://127.0.0.1:" + str(server.server_address[1])

server, base = start_server(Handler)

# your code here


### Q10 · Build

POST `order` as JSON to `/api/orders`. The server replies `201 Created`
with the stored order, including a new `id`. Print the status code, the
new id and the item. Shut the server down at the end.

**Expected output:**

```text
201
101
apple
```

<details><summary><b>Solution</b></summary>

```python
request = urllib.request.Request(
    base + "/api/orders", data=json.dumps(order).encode(),
    headers={"Content-Type": "application/json"})
with urllib.request.urlopen(request, timeout=2) as response:
    created = json.load(response)
    print(response.status)
print(created["id"])
print(created["item"])
server.shutdown()
server.server_close()
```

</details>

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

order = {"item": "apple", "qty": 3}

class Handler(BaseHTTPRequestHandler):
    def do_POST(self):
        size = int(self.headers["Content-Length"])
        order = json.loads(self.rfile.read(size))
        order["id"] = 101
        self.send_json(201, order)

    def send_json(self, status, data, headers=None):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, "http://127.0.0.1:" + str(server.server_address[1])

server, base = start_server(Handler)

# your code here


---

## Small real tasks · 5 to 8 lines

### Q11 · Real-life

A health checker visits three paths. For each one print the path and a
verdict: the user's name when the call works, `missing` for a 404 (the
client asked for something that does not exist), and `server error, try
later` for any 5xx. Shut the server down at the end.

**Expected output:**

```text
/api/users/1: Ana
/api/users/99: missing
/api/crash: server error, try later
```

<details><summary><b>Solution</b></summary>

```python
for path in paths:
    try:
        with urllib.request.urlopen(base + path, timeout=2) as response:
            verdict = json.load(response)["name"]
    except urllib.error.HTTPError as err:
        if err.code == 404:
            verdict = "missing"
        elif err.code >= 500:
            verdict = "server error, try later"
        else:
            raise
    print(path + ": " + verdict)
server.shutdown()
server.server_close()
```

</details>

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

paths = ["/api/users/1", "/api/users/99", "/api/crash"]

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/api/users/1":
            self.send_json(200, {"name": "Ana"})
        elif self.path == "/api/crash":
            self.send_json(500, {"error": "boom"})
        else:
            self.send_json(404, {"error": "not found"})

    def send_json(self, status, data, headers=None):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, "http://127.0.0.1:" + str(server.server_address[1])

server, base = start_server(Handler)

# your code here


### Q12 · Real-life

The server at `base` answers `503` for its first two requests and works
from the third on. Write `get_with_retry(url, attempts=4, delay=0.01)`: on
a 5xx error wait `delay`, then double the wait each time before trying
again; any other error, or running out of attempts, is raised. Return the
JSON data, the attempt number that worked, and the list of waits used.
Call it on `base`, print the three results, and shut the server down.

**Expected output:**

```text
{'data': 'ready'}
3
[0.01, 0.02]
```

<details><summary><b>Solution</b></summary>

```python
def get_with_retry(url, attempts=4, delay=0.01):
    waits = []
    for attempt in range(1, attempts + 1):
        try:
            with urllib.request.urlopen(url, timeout=2) as response:
                return json.load(response), attempt, waits
        except urllib.error.HTTPError as err:
            if err.code < 500 or attempt == attempts:
                raise
            waits.append(delay * 2 ** (attempt - 1))
            time.sleep(waits[-1])

data, attempt, waits = get_with_retry(base)
print(data)
print(attempt)
print(waits)
server.shutdown()
server.server_close()
```

</details>

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    hits = 0

    def do_GET(self):
        Handler.hits += 1
        if Handler.hits <= 2:
            self.send_json(503, {"error": "busy"})
        else:
            self.send_json(200, {"data": "ready"})

    def send_json(self, status, data, headers=None):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, "http://127.0.0.1:" + str(server.server_address[1])

server, base = start_server(Handler)

# your code here


### Q13 · Real-life

`/slow` never answers in time, `/fast` answers at once. Write
`fetch(path)` that asks for a path with a **0.3-second timeout** and
returns the `message` from the JSON, or the text `timed out` when the wait
runs out. Print the result for `/fast` and for `/slow`. Then release the
stuck handler and shut the server down.

**Expected output:**

```text
quick reply
timed out
```

<details><summary><b>Solution</b></summary>

```python
def fetch(path):
    try:
        with urllib.request.urlopen(base + path, timeout=0.3) as response:
            return json.load(response)["message"]
    except (TimeoutError, urllib.error.URLError):
        return "timed out"

print(fetch("/fast"))
print(fetch("/slow"))
release.set()
server.shutdown()
server.server_close()
```

</details>

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

release = threading.Event()

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/fast":
            self.send_json(200, {"message": "quick reply"})
        else:
            release.wait(5)  # hold the request until the notebook lets go

    def send_json(self, status, data, headers=None):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, "http://127.0.0.1:" + str(server.server_address[1])

server, base = start_server(Handler)

# your code here


### Q14 · Real-life

`/api/items?page=N` returns two items per page plus a `next` field: the
next page number, or `null` when there are no more. Follow the pages from
page 1 (build each URL with `urlencode`), collect every item, and print how
many there are and their names in one list. Shut the server down at the end.

**Expected output:**

```text
5
['pen', 'ink', 'pad', 'tape', 'glue']
```

<details><summary><b>Solution</b></summary>

```python
names = []
page = 1
while page:
    url = base + "/api/items?" + urllib.parse.urlencode({"page": page})
    with urllib.request.urlopen(url, timeout=2) as response:
        data = json.load(response)
    names.extend(data["items"])
    page = data["next"]
print(len(names))
print(names)
server.shutdown()
server.server_close()
```

</details>

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

ITEMS = ["pen", "ink", "pad", "tape", "glue"]

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        parts = urllib.parse.urlsplit(self.path)
        page = int(urllib.parse.parse_qs(parts.query).get("page", ["1"])[0])
        start = (page - 1) * 2
        more = start + 2 < len(ITEMS)
        self.send_json(200, {"items": ITEMS[start:start + 2], "next": page + 1 if more else None})

    def send_json(self, status, data, headers=None):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, "http://127.0.0.1:" + str(server.server_address[1])

server, base = start_server(Handler)

# your code here


### Q15 · Real-life

The server at `base` only serves `/api/secret` to callers that send the
header `Authorization: Bearer s3cret`; otherwise it answers 401. Call it
without the header and print `no token: ` plus the status code, then call
it with the header and print `with token: ` plus the `message` from the
JSON. Shut the server down at the end.

**Expected output:**

```text
no token: 401
with token: the vault code is 1234
```

<details><summary><b>Solution</b></summary>

```python
url = base + "/api/secret"
try:
    urllib.request.urlopen(url, timeout=2)
except urllib.error.HTTPError as err:
    print("no token:", err.code)
request = urllib.request.Request(url, headers={"Authorization": "Bearer s3cret"})
with urllib.request.urlopen(request, timeout=2) as response:
    print("with token:", json.load(response)["message"])
server.shutdown()
server.server_close()
```

</details>

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.headers.get("Authorization") == "Bearer s3cret":
            self.send_json(200, {"message": "the vault code is 1234"})
        else:
            self.send_json(401, {"error": "unauthorized"}, {"WWW-Authenticate": "Bearer"})

    def send_json(self, status, data, headers=None):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, "http://127.0.0.1:" + str(server.server_address[1])

server, base = start_server(Handler)

# your code here


---

## Puzzles & builds · 8 to 15 lines

### Q16 · Puzzle

This is what a server really receives, as text. Parse `raw` by hand:
print the method, the path without its query, the query as a dict of
single values, then every header as `name=value` with lower-case names in
alphabetical order, then the `item` from the JSON body, and finally
whether the `Content-Length` header equals the real size of the body in
bytes. Lines of an HTTP message end in `\r\n`, and a blank line separates
the headers from the body.

**Expected output:**

```text
POST
/api/orders
{'id': '7', 'rush': 'yes'}
content-length=16
content-type=application/json
host=shop.example.com
pear
True
```

<details><summary><b>Solution</b></summary>

```python
head, _, body = raw.partition("\r\n\r\n")
lines = head.split("\r\n")
method, target, version = lines[0].split(" ")
parts = urlsplit(target)
query = {key: values[0] for key, values in parse_qs(parts.query).items()}
headers = {}
for line in lines[1:]:
    name, _, value = line.partition(": ")
    headers[name.lower()] = value
print(method)
print(parts.path)
print(query)
for name in sorted(headers):
    print(name + "=" + headers[name])
print(json.loads(body)["item"])
print(int(headers["content-length"]) == len(body.encode()))
```

</details>

In [ ]:
raw = ("POST /api/orders?id=7&rush=yes HTTP/1.1\r\n"
       "Host: shop.example.com\r\n"
       "Content-Type: application/json\r\n"
       "Content-Length: 16\r\n"
       "\r\n"
       '{"item": "pear"}')
import json
from urllib.parse import parse_qs, urlsplit

# your code here


### Q17 · Build

Write your own tiny web server. A `Handler` answers `GET /hello` with
status 200, content type `text/plain` and the body `hello`; any other GET
path with a 404 (`self.send_error(404)`). Give it a `log_message` that
does nothing, and start it on `127.0.0.1` with port `0` in a thread. Then,
with a timeout, print the status and body for `/hello`, the status for
`/nope`, and the status of a `DELETE` to `/hello` (your handler has no
`do_DELETE`, so the server itself must answer). Shut the server down.

**Expected output:**

```text
200 hello
404
501
```

<details><summary><b>Solution</b></summary>

```python
class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/hello":
            body = b"hello"
            self.send_response(200)
            self.send_header("Content-Type", "text/plain")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)
        else:
            self.send_error(404)

    def log_message(self, format, *args):
        pass

server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
base = "http://127.0.0.1:" + str(server.server_address[1])

with urllib.request.urlopen(base + "/hello", timeout=2) as response:
    print(response.status, response.read().decode())

def status_of(request):
    try:
        with urllib.request.urlopen(request, timeout=2) as response:
            return response.status
    except urllib.error.HTTPError as err:
        return err.code

print(status_of(base + "/nope"))
print(status_of(urllib.request.Request(base + "/hello", method="DELETE")))
server.shutdown()
server.server_close()
```

</details>

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

# your code here


### Q18 · Puzzle

Invent a text protocol over a raw socket: one command per line, one reply
per line. `SET key value` replies `OK`, `GET key` replies the value or
`NOT FOUND`, `QUIT` replies `BYE` and ends the conversation. Write
`serve_once()` that accepts a single client on `listener` and keeps its
own dict (use `conn.makefile("rw")` to read and write lines, and flush
after replying), start it in a thread, then connect as the client with a
2-second timeout and send every command in `commands`, printing each
command and its reply like `GET color -> blue`. Close the listener at the end.

**Expected output:**

```text
SET color blue -> OK
GET color -> blue
GET size -> NOT FOUND
QUIT -> BYE
```

<details><summary><b>Solution</b></summary>

```python
def serve_once():
    conn, _ = listener.accept()
    with conn, conn.makefile("rw", encoding="utf-8", newline="\n") as stream:
        store = {}
        for line in stream:
            command, _, rest = line.strip().partition(" ")
            if command == "SET":
                key, _, value = rest.partition(" ")
                store[key] = value
                reply = "OK"
            elif command == "GET":
                reply = store.get(rest, "NOT FOUND")
            else:
                reply = "BYE"
            stream.write(reply + "\n")
            stream.flush()
            if reply == "BYE":
                break

threading.Thread(target=serve_once, daemon=True).start()
client = socket.create_connection(address, timeout=2)
with client, client.makefile("rw", encoding="utf-8", newline="\n") as stream:
    for command in commands:
        stream.write(command + "\n")
        stream.flush()
        print(command, "->", stream.readline().strip())
listener.close()
```

</details>

In [ ]:
import socket
import threading

listener = socket.socket()
listener.bind(("127.0.0.1", 0))
listener.listen()
address = listener.getsockname()

commands = ["SET color blue", "GET color", "GET size", "QUIT"]

# your code here


### Q19 · Real-life

A dashboard needs the names of items 1 to 5 from `/api/items/N`. Each
request takes about 0.2 seconds on the server, so fetching them one by one
would take a full second. Fetch them concurrently with a
`ThreadPoolExecutor` of 5 workers and a request timeout, and print the
names in id order. Then print `True` if the whole thing took under 0.8
seconds. Shut the server down at the end.

**Expected output:**

```text
['pen', 'ink', 'pad', 'tape', 'glue']
True
```

<details><summary><b>Solution</b></summary>

```python
from concurrent.futures import ThreadPoolExecutor
import time

def fetch_name(number):
    with urllib.request.urlopen(base + "/api/items/" + str(number), timeout=2) as response:
        return json.load(response)["name"]

started = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    names = list(pool.map(fetch_name, range(1, 6)))
elapsed = time.perf_counter() - started
print(names)
print(elapsed < 0.8)
server.shutdown()
server.server_close()
```

</details>

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

NAMES = ["pen", "ink", "pad", "tape", "glue"]

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        time.sleep(0.2)
        number = int(self.path.rsplit("/", 1)[1])
        self.send_json(200, {"id": number, "name": NAMES[number - 1]})

    def send_json(self, status, data, headers=None):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, "http://127.0.0.1:" + str(server.server_address[1])

server, base = start_server(Handler)

# your code here


---

## Q20 · Story project: Todo Service

You are building a tiny REST service for a to-do app, and a client to talk
to it, both inside this notebook. `GET /todos` lists, `GET /todos/<id>`
reads one, `POST /todos` creates (JSON body with a `title`), and
`DELETE /todos/<id>` removes. Each step builds on the one before. The
server keeps its data in the `todos` dictionary.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

todos = {}  # id -> {"id": ..., "title": ..., "done": False}

def send_json(handler, status, data):
    body = json.dumps(data).encode()
    handler.send_response(status)
    handler.send_header("Content-Type", "application/json")
    handler.send_header("Content-Length", str(len(body)))
    handler.end_headers()
    handler.wfile.write(body)

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, "http://127.0.0.1:" + str(server.server_address[1])


### Step 1

Write `TodoHandler(BaseHTTPRequestHandler)` with a `do_GET`: `/todos` replies 200 with the list of todos ordered by id, `/todos/<id>` replies 200 with that todo, and anything unknown (including an id that does not exist) replies 404 with `{"error": "not found"}`. Use `send_json(self, ...)` and a silent `log_message`. Start it with `start_server` and print whether `base` starts with `http://127.0.0.1:`.

**Expected output:**

```text
True
```

<details><summary><b>Solution</b></summary>

```python
class TodoHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/todos":
            send_json(self, 200, [todos[key] for key in sorted(todos)])
        elif self.path.startswith("/todos/"):
            todo_id = self.path.removeprefix("/todos/")
            if todo_id.isdigit() and int(todo_id) in todos:
                send_json(self, 200, todos[int(todo_id)])
            else:
                send_json(self, 404, {"error": "not found"})
        else:
            send_json(self, 404, {"error": "not found"})

    def log_message(self, format, *args):
        pass

server, base = start_server(TodoHandler)
print(base.startswith("http://127.0.0.1:"))
```

</details>

In [ ]:
# your code here


### Step 2

Write the client helper `call(method, path, body=None)`. It sends JSON (when there is a body) to `base + path` with a 2-second timeout and returns `(status, data)`, where `data` is the decoded JSON reply or `None` when the reply is empty. A 4xx or 5xx must not crash it: it returns that status and the error body instead. Print `call("GET", "/todos")` and `call("GET", "/todos/5")`.

**Expected output:**

```text
(200, [])
(404, {'error': 'not found'})
```

<details><summary><b>Solution</b></summary>

```python
def call(method, path, body=None):
    data = json.dumps(body).encode() if body is not None else None
    request = urllib.request.Request(base + path, data=data, method=method,
                                     headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=2) as response:
            status, raw = response.status, response.read()
    except urllib.error.HTTPError as err:
        status, raw = err.code, err.read()
    return status, (json.loads(raw) if raw else None)

print(call("GET", "/todos"))
print(call("GET", "/todos/5"))
```

</details>

In [ ]:
# your code here


### Step 3

The server cannot create or delete yet. Make `WriteHandler(TodoHandler)` that adds `do_POST` and `do_DELETE`. `POST /todos` reads the JSON body; if it has no non-empty `title` reply 400 with `{"error": "title required"}`, otherwise store `{"id": new_id, "title": ..., "done": False}` (the new id is one more than the largest id so far) and reply 201 with it. `DELETE /todos/<id>` removes the todo and replies 204 with no body, or 404 if it does not exist. Shut the old server down, start one with `WriteHandler`, and print whether the new `base` starts with `http://127.0.0.1:`.

**Expected output:**

```text
True
```

<details><summary><b>Solution</b></summary>

```python
class WriteHandler(TodoHandler):
    def do_POST(self):
        size = int(self.headers.get("Content-Length", 0))
        try:
            payload = json.loads(self.rfile.read(size))
        except json.JSONDecodeError:
            payload = {}
        title = payload.get("title") if isinstance(payload, dict) else None
        if not title:
            send_json(self, 400, {"error": "title required"})
            return
        new_id = max(todos, default=0) + 1
        todos[new_id] = {"id": new_id, "title": title, "done": False}
        send_json(self, 201, todos[new_id])

    def do_DELETE(self):
        todo_id = self.path.removeprefix("/todos/")
        if todo_id.isdigit() and int(todo_id) in todos:
            del todos[int(todo_id)]
            self.send_response(204)
            self.send_header("Content-Length", "0")
            self.end_headers()
        else:
            send_json(self, 404, {"error": "not found"})

server.shutdown()
server.server_close()
server, base = start_server(WriteHandler)
print(base.startswith("http://127.0.0.1:"))
```

</details>

In [ ]:
# your code here


### Step 4

Exercise it through `call`. Create `Buy milk` and `Walk dog`, printing each status and new id. Try a POST with an empty body object `{}` and print its status and error. Delete todo 1 and print the status, delete it again and print that status, then print the titles that are left. Keep each status in a variable; you need them next.

**Expected output:**

```text
201 1
201 2
400 title required
204
404
['Walk dog']
```

<details><summary><b>Solution</b></summary>

```python
first_status, first = call("POST", "/todos", {"title": "Buy milk"})
second_status, second = call("POST", "/todos", {"title": "Walk dog"})
print(first_status, first["id"])
print(second_status, second["id"])
bad_status, bad = call("POST", "/todos", {})
print(bad_status, bad["error"])
delete_status, _ = call("DELETE", "/todos/1")
again_status, _ = call("DELETE", "/todos/1")
print(delete_status)
print(again_status)
_, remaining = call("GET", "/todos")
print([todo["title"] for todo in remaining])
```

</details>

In [ ]:
# your code here


### Step 5

Wrap up. Using the statuses from the last step, count the successful creates (201) and the rejected requests (any 4xx among the bad POST and the repeated DELETE). Print `2 created, 2 rejected, 1 left` built from those numbers and the list you fetched, then shut the server down and print `server stopped`.

**Expected output:**

```text
2 created, 2 rejected, 1 left
server stopped
```

<details><summary><b>Solution</b></summary>

```python
statuses = [first_status, second_status, bad_status, again_status]
created = statuses.count(201)
rejected = len([status for status in statuses if 400 <= status < 500])
print(str(created) + " created, " + str(rejected) + " rejected, " + str(len(remaining)) + " left")
server.shutdown()
server.server_close()
print("server stopped")
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB31 · Real World Programs** in `advanced/`.